# Notebook 09B — NCCS Model Optimization and Ranking Evaluation

**Objective:** Improve the ranking of filings with self-reported significant diversion of assets while preserving Notebook 09 as the baseline. A positive disclosure is **not** verified fraud.

**Design:** Train on tax years **2010–2020**, select exclusively on **2021–2022 validation**, and **do not inspect 2023–2024 labels or performance** in this notebook. Since the prior work has already examined the 2023–2024 holdout, it is no longer a pristine independent test for subsequent model development. Results here are validation experiments, not unbiased future-performance claims.

**Outputs:** `data/processed/notebook09b/`. Does not overwrite Notebook 09 or 09A artifacts. Start from the project root or `notebooks/` and use the `irs990` kernel. Estimated runtime and RAM vary; begin with `RUN_EXTENDED = False`.

## 1. Paths, dependencies, and baseline metadata

In [ ]:
from pathlib import Path
import json, gc, time
import numpy as np
import pandas as pd
import duckdb
import sklearn
from IPython.display import display

SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT = ROOT / "data/processed/nccs_longitudinal_features_v1.parquet"
BASELINE_DIR = ROOT / "data/processed/notebook09"
OUT = ROOT / "data/processed/notebook09b"
OUT.mkdir(parents=True, exist_ok=True)
assert INPUT.is_file(), f"Missing feature file: {INPUT}"
manifest_path = BASELINE_DIR / "model_manifest.json"
assert manifest_path.is_file(), f"Missing Notebook 09 manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text())
FEATURES = list(manifest["features"])
assert manifest["train_years"] == [2010, 2020]
assert manifest["validation_years"] == [2021, 2022]
assert "fraud_label" not in FEATURES

RUN_EXTENDED = True  # Extended experiments enabled; requires additional CPU, RAM, and time.
con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET memory_limit = '6GB'")
con.execute(f"""
    CREATE VIEW features AS
    SELECT *
    FROM read_parquet('{INPUT.as_posix().replace("'", "''")}')
""")
cols = set(con.execute("DESCRIBE features").df().column_name)
assert set(FEATURES + ["OBJECTID", "ein", "tax_year", "fraud_label"]) <= cols
print("Project:", ROOT)
print("Feature file:", INPUT)
print("Output:", OUT)
print("sklearn:", sklearn.__version__)
print("Features:", len(FEATURES))
print("Extended experiments:", RUN_EXTENDED)

## 2. Load only development years and validate the split

Do not load 2023–2024 labels for tuning. Missing values remain `NaN` until preprocessing inside a fitted pipeline.

In [ ]:
SELECT_COLS = ["OBJECTID", "ein", "tax_year", "fraud_label"] + FEATURES
quoted = ", ".join('"' + x.replace('"','""') + '"' for x in SELECT_COLS)

def load_split(lo, hi):
    df = con.execute(f"""
        SELECT {quoted}
        FROM features
        WHERE tax_year BETWEEN ? AND ? AND fraud_label IN (0,1)
        ORDER BY tax_year, ein, OBJECTID
    """, [lo, hi]).df()
    for col in FEATURES:
        df[col] = (pd.to_numeric(df[col], errors="coerce")
                   .replace([np.inf, -np.inf], np.nan).astype("float32"))
    assert df.fraud_label.isin([0,1]).all()
    assert not df.duplicated(["ein", "tax_year"]).any()
    return df

train = load_split(2010, 2020)
val = load_split(2021, 2022)
y_val = val.fraud_label.astype("int8").to_numpy()
print("Train:", len(train), "positives:", int(train.fraud_label.sum()))
print("Validation:", len(val), "positives:", int(y_val.sum()))
assert train.fraud_label.nunique() == 2 and len(np.unique(y_val)) == 2
assert set(train.tax_year).isdisjoint(set(val.tax_year))
assert val.tax_year.between(2021,2022).all()

## 3. Ranking metrics

**Primary metric:** validation Average Precision (AP). **Secondary:** positive disclosures in top 500 and top 1,000, ROC-AUC, and per-year stability. We avoid optimizing only top 100 because rare positives make that count noisy. All candidates are evaluated on **exactly the same full validation population**. Scores are ranking scores, not calibrated fraud probabilities.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def ranking_metrics(y, scores, ks=(100,250,500,1000)):
    y = np.asarray(y, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    assert len(y) == len(scores) and np.isfinite(scores).all()
    out = {
        "n": len(y), "positives": int(y.sum()), "prevalence": float(y.mean()),
        "average_precision": float(average_precision_score(y, scores)),
        "roc_auc": float(roc_auc_score(y, scores)) if np.unique(y).size == 2 else np.nan,
    }
    order = np.argsort(-scores, kind="stable")
    for k in ks:
        count = min(k, len(y))
        found = int(y[order[:count]].sum())
        out[f"found_at_{k}"] = found
        out[f"precision_at_{k}"] = found / count if count else np.nan
    return out

# Note: sklearn AP is NOT the same as trapezoidal PR-AUC reported in Notebook 09.
print("Validation random-ranking AP baseline (prevalence):", round(y_val.mean(), 7))

## 4. Model candidates and feature groups

The full feature list comes directly from Notebook 09. We test whether removing historical features improves ranking. Training downsampling retains **all positive examples** and samples negatives deterministically; this changes class prevalence and means output scores are not calibrated probabilities.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

HISTORICAL_FEATURES = [
    "rev_growth_3yr", "asset_growth_3yr", "rev_volatility_3yr",
    "avg_net_income_3yr", "flag_board_independence_drop",
    "flag_sustained_deficit", "has_three_consecutive_years",
]
FEATURE_SETS = {
    "all_features": FEATURES,
    "current_year_only": [f for f in FEATURES if f not in HISTORICAL_FEATURES],
}
assert FEATURE_SETS["current_year_only"]

def sampled_training(df, cap, seed=SEED):
    pos = df.loc[df.fraud_label.eq(1)]
    neg = df.loc[df.fraud_label.eq(0)]
    if cap < len(pos):
        raise ValueError("Training cap smaller than positive count")
    n_neg = min(len(neg), cap - len(pos))
    sampled_neg = neg.sample(n=n_neg, random_state=seed) if n_neg < len(neg) else neg
    return pd.concat([pos, sampled_neg], ignore_index=True).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

def make_model(kind):
    imputer = SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)
    if kind == "sgd_logistic":
        return Pipeline([
            ("impute", imputer), ("scale", StandardScaler()),
            ("model", SGDClassifier(loss="log_loss", alpha=0.0001,
                class_weight="balanced", max_iter=1500, tol=1e-3, random_state=SEED))])
    if kind == "regularized_logistic":
        return Pipeline([
            ("impute", imputer), ("scale", RobustScaler()),
            ("model", LogisticRegression(C=0.1, class_weight="balanced",
                max_iter=400, solver="liblinear", random_state=SEED))])
    if kind == "hgb_baseline":
        return Pipeline([
            ("impute", imputer),
            ("model", HistGradientBoostingClassifier(max_iter=160, max_leaf_nodes=15,
                learning_rate=0.06, l2_regularization=2.0, random_state=SEED))])
    if kind == "hgb_regularized":
        return Pipeline([
            ("impute", imputer),
            ("model", HistGradientBoostingClassifier(max_iter=220, max_leaf_nodes=7,
                min_samples_leaf=60, learning_rate=0.04,
                l2_regularization=10.0, random_state=SEED))])
    raise ValueError(kind)

# Keep the first pass modest. Extended mode adds larger training samples.
EXPERIMENTS = [
    ("sgd_logistic", "all_features", 500_000),
    ("regularized_logistic", "all_features", 500_000),
    ("regularized_logistic", "current_year_only", 500_000),
    ("hgb_baseline", "all_features", 250_000),
    ("hgb_regularized", "all_features", 250_000),
    ("hgb_regularized", "current_year_only", 250_000),
]
if RUN_EXTENDED:
    EXPERIMENTS += [
        ("regularized_logistic", "all_features", 1_000_000),
        ("regularized_logistic", "all_features", len(train)),
        ("hgb_regularized", "all_features", 500_000),
    ]
print("Experiments:", len(EXPERIMENTS))
display(pd.DataFrame(EXPERIMENTS, columns=["model", "feature_set", "train_cap"]))

## 5. Fit candidates and compare on validation only

This cell can take several minutes or longer. It saves results after each candidate so interrupted runs retain progress. If RAM becomes constrained, keep `RUN_EXTENDED = False`. **No test labels are accessed.**

In [ ]:
results = []
validation_scores = {}
for i, (kind, feature_set, cap) in enumerate(EXPERIMENTS, start=1):
    name = f"{kind}__{feature_set}__n{cap}"
    cols_used = FEATURE_SETS[feature_set]
    fit_df = sampled_training(train, cap)
    model = make_model(kind)
    started = time.monotonic()
    print(f"[{i}/{len(EXPERIMENTS)}] Fitting {name} | rows={len(fit_df):,}", flush=True)
    model.fit(fit_df[cols_used], fit_df.fraud_label.astype("int8"))
    scores = model.predict_proba(val[cols_used])[:, 1]
    elapsed = round(time.monotonic() - started, 1)
    validation_scores[name] = np.asarray(scores, dtype="float32")
    row = {
        "experiment": name, "model": kind, "feature_set": feature_set,
        "train_rows": len(fit_df), "train_positives": int(fit_df.fraud_label.sum()),
        "elapsed_seconds": elapsed,
        **ranking_metrics(y_val, scores),
    }
    results.append(row)
    pd.DataFrame(results).sort_values("average_precision", ascending=False).to_csv(
        OUT / "01_validation_experiment_comparison.csv", index=False)
    print("  AP:", round(row["average_precision"], 6),
          "found@500:", row["found_at_500"],
          "found@1000:", row["found_at_1000"], flush=True)
    del fit_df, model, scores
    gc.collect()

comparison = pd.DataFrame(results).sort_values(
    ["average_precision", "found_at_1000", "found_at_500"],
    ascending=[False, False, False]
).reset_index(drop=True)
display(comparison)
SELECTED = comparison.iloc[0]["experiment"]
print("Selected by validation AP:", SELECTED)

## 6. Validate stability across 2021 and 2022

A candidate can score well in aggregate but fail in one year. This table reports per-year AP and top-K counts for every candidate. These are **validation diagnostics**; do not use the 2023–2024 holdout for model choice.

In [ ]:
year_rows = []
for name, scores in validation_scores.items():
    for year in (2021, 2022):
        mask = val.tax_year.to_numpy() == year
        year_rows.append({"experiment": name, "tax_year": year,
                          **ranking_metrics(y_val[mask], scores[mask])})
year_results = pd.DataFrame(year_rows).sort_values(["experiment", "tax_year"])
display(year_results[["experiment", "tax_year", "n", "positives",
                      "average_precision", "roc_auc", "found_at_500", "found_at_1000"]])
year_results.to_csv(OUT / "02_validation_by_year.csv", index=False)

## 7. Feature missingness and training/scoring population shift

Compare **2024 labeled vs unknown-label filings** without treating unknown labels as negatives or calculating accuracy for them. The 2024 queue from Notebook 09 provides baseline scores for descriptive comparison. These comparisons do not select a model and do not use 2024 outcomes.

In [ ]:
# Aggregate feature missingness directly in DuckDB; avoid loading 2024 predictors into RAM.
feature_missing_rows = []
for feature in FEATURES:
    q = '"' + feature.replace('"','""') + '"'
    rows = con.execute(f"""
        SELECT CASE WHEN fraud_label IN (0,1) THEN 'known' ELSE 'unknown' END AS label_status,
               COUNT(*) AS filings,
               COUNT(*) FILTER (WHERE {q} IS NULL) AS missing
        FROM features
        WHERE tax_year = 2024
        GROUP BY 1
    """).df()
    for _, r in rows.iterrows():
        feature_missing_rows.append({"feature": feature, "label_status": r.label_status,
                                     "filings": int(r.filings), "missing": int(r.missing),
                                     "missing_pct": 100.0 * r.missing / r.filings})
missingness = pd.DataFrame(feature_missing_rows)
missing_pivot = missingness.pivot(index="feature", columns="label_status", values="missing_pct")
missing_pivot["unknown_minus_known_pp"] = missing_pivot.get("unknown", 0) - missing_pivot.get("known", 0)
display(missing_pivot.sort_values("unknown_minus_known_pp", ascending=False))
missingness.to_csv(OUT / "03_2024_known_vs_unknown_missingness.csv", index=False)

queue_path = BASELINE_DIR / "09_all_2024_investigation_queue.parquet"
if queue_path.is_file():
    queue = pd.read_parquet(queue_path, columns=["fraud_label", "risk_score", "label_status"])
    score_summary = queue.groupby("label_status").risk_score.agg(
        ["count", "median", "mean", "min", "max"])
    display(score_summary)
    score_summary.to_csv(OUT / "04_notebook09_score_distribution_by_label_status.csv")
    print("NOTE: Score comparisons are descriptive; unknown is not a negative label.")
    del queue
else:
    print("Notebook 09 queue unavailable; skipping score distribution comparison:", queue_path)

## 8. Compare against Notebook 09 validation baseline (if present)

This is an *informational* comparison. Model selection above uses the same 2021–2022 validation cohort. If Notebook 09 used different training caps or preprocessing, those settings are documented in its manifest.

In [ ]:
baseline_csv = BASELINE_DIR / "03_validation_model_comparison.csv"
if baseline_csv.is_file():
    previous = pd.read_csv(baseline_csv)
    print("Notebook 09 validation results:")
    display(previous[[c for c in ["model", "n", "positives", "average_precision",
                                   "roc_auc", "found_at_500", "found_at_1000"] if c in previous.columns]])
else:
    print("No Notebook 09 validation comparison found:", baseline_csv)
print("Notebook 09B selected:", SELECTED)
display(comparison.head(3)[["experiment", "average_precision", "roc_auc",
                            "found_at_500", "found_at_1000"]])

## 9. Export experiment manifest and recommendation

No selected model is refitted or deployed here. Notebook 09 remains the production baseline. If validation gains are meaningful and stable, a subsequent notebook can refit the selected configuration and establish a genuinely independent evaluation strategy before operational use.

In [ ]:
winner = comparison.iloc[0]
report = {
    "source_parquet": str(INPUT),
    "baseline_manifest": str(manifest_path),
    "train_years": [2010, 2020],
    "validation_years": [2021, 2022],
    "holdout_years_not_used_for_selection": [2023, 2024],
    "selection_metric": "validation_average_precision",
    "winner": str(SELECTED),
    "winner_average_precision": float(winner.average_precision),
    "winner_found_at_500": int(winner.found_at_500),
    "winner_found_at_1000": int(winner.found_at_1000),
    "seed": SEED,
    "extended": RUN_EXTENDED,
    "features": FEATURES,
    "experiment_count": len(EXPERIMENTS),
    "scores_are_calibrated_probabilities": False,
    "target": "Self-reported significant diversion of assets, not verified fraud",
    "caveats": [
        "Training uses negative downsampling and all available positives",
        "The 2023–2024 holdout was already inspected in prior work and is not pristine",
        "Unknown labels are not negative labels and are excluded from supervised metrics",
        "Point-in-time filing availability has not been established",
        "Repeated validation experimentation can overfit the validation years",
    ],
}
(OUT / "05_experiment_manifest.json").write_text(json.dumps(report, indent=2))
print("Selected experiment:", SELECTED)
print("Validation AP:", round(float(winner.average_precision), 6))
print("Top 500 disclosures:", int(winner.found_at_500))
print("Top 1000 disclosures:", int(winner.found_at_1000))
print("Saved outputs in:", OUT)
print("Next: review validation metrics, year stability, and known-vs-unknown missingness before any model deployment.")

## Interpretation and next steps

### 1. Model optimization results

Notebook 09B completed nine model experiments using the 2010–2020 training period and 2021–2022 validation period.

The highest validation Average Precision was achieved by **regularized Logistic Regression using all 18 features and a 1,000,000-record training sample**.

| Metric | Notebook 09 Gradient Boosting | Notebook 09B Logistic Regression |
|---|---:|---:|
| Average Precision | 0.002251 | **0.003985** |
| ROC-AUC | **0.664204** | 0.5996 |
| Positive disclosures in top 500 | 7 | **11** |
| Positive disclosures in top 1,000 | 11 | 11 |

The optimized Logistic Regression achieved approximately 77% higher validation Average Precision and identified four additional positive disclosures among the top 500 ranked filings.

However, its lower ROC-AUC indicates weaker overall discrimination across the full validation population.

**These are development results, not evidence of improved real-world fraud detection.** The positive label represents self-reported significant diversion of assets, not confirmed fraud.

### 2. Training sample size and feature selection

Increasing the training sample from 500,000 to 1,000,000 records improved the strongest Logistic Regression configuration. Increasing it further to approximately 2.46 million records did not produce additional improvement.

Experiments using only current-year features also produced competitive results, suggesting that the historical features may provide limited incremental predictive value.

Further analysis is needed before concluding that historical features should be removed.

### 3. Logistic Regression convergence

Several Logistic Regression experiments generated convergence warnings, indicating that the optimizer may not have reached a stable solution.

The highest-performing configuration should therefore be treated as a provisional candidate.

**Notebook 09C will investigate solver configurations, iteration limits, convergence, and ranking stability before selecting a final model.**

### 4. Missingness differences between labeled and unlabeled filings

The 2024 missingness analysis identified a major structural difference between the two populations:

- **355,973 filings** have known disclosure labels.
- **205,447 filings** have unknown disclosure labels.
- All unknown-label filings are missing total assets, liabilities, and governance information in the current feature matrix.
- Known-label filings have complete values for these fields.

This suggests that label availability and financial/governance feature availability may be linked to the underlying NCCS source structure or extraction process.

Consequently, model performance measured on labeled filings cannot automatically be generalized to unlabeled filings.

The existing 2024 investigation queue should remain exploratory until this difference is understood.

### 5. Model evaluation and limitations

Model selection must continue to use the 2021–2022 validation period rather than optimizing against the previously examined 2023–2024 holdout.

Average Precision should be evaluated alongside precision and positive-disclosure counts at the top 100, 500, and 1,000.

Small differences in top-K performance should be interpreted cautiously because positive disclosures are extremely rare.

The 2023–2024 period can no longer serve as an untouched independent test for new modeling decisions informed by its previously examined results.

### 6. Next steps

**Notebook 09C — Missingness Investigation and Model Stabilization**

1. Determine why unknown-label filings systematically lack financial and governance fields.
2. Investigate whether these filings require a separate modeling or scoring approach.
3. Resolve Logistic Regression convergence warnings.
4. Reevaluate the strongest configurations using validation Average Precision, top-K metrics, and year-by-year stability.
5. Recommend whether to retain Gradient Boosting, adopt Logistic Regression, or develop separate scoring approaches.

**Notebook 10 — Peer Benchmarking and Risk Explanations**

After resolving the major limitations identified in Notebook 09C, develop peer-group comparisons, feature attribution, and interpretable investigation summaries.

### Final conclusion

Notebook 09B identified a promising improvement in validation ranking performance, but the results are not sufficient to justify replacing Notebook 09's existing model.